# Dicts inside containers: two options

**A. As built.** Dicts inside a list, tuple or dict keep the keys they were built with. Writes inside them are
never tracked; only writes to a dict's own name change its keys. One rule, no exceptions; the cost is stale types
after a write inside.

**B. Tracked where possible** (today, plus `append` joining keys). Every case follows a rule plain dicts already have:
writes the checker can pin to one dict update it, writes it can't pin make it plain, adding a dict joins keys
(like `if` branches), writes it can't see leave it stale.

Each line shows `A:` and `B:`. Hover shows today's build, which is B except where a line says *(B: not built yet)*.


In [ ]:
# 1. Building: the same in both
rows = [{'loss': 0.4, 'tag': 'a'}, {'loss': 0.5, 'tag': 'b'}]
rows  # A and B: list[dict[{'loss': float, 'tag': str}]]
rows[0]['loss']  # A and B: float

In [ ]:
# 2. Adding a dict
added = [{'loss': 0.4, 'tag': 'a'}]
added.append({'loss': 0.6, 'tag': 'c'})  # same keys
added  # A: list[dict[{'loss': float, 'tag': str}]]   B: the same (keys joined)  (B: not built yet, hover shows plain)

other = [{'loss': 0.4, 'tag': 'a'}]
other.append({'loss': 0.6, 'note': 'x'})  # other keys
other  # A: list[dict[{'loss': float, 'tag': str}]] (stale: the new dict has 'note', not 'tag')
#        B: list[dict[{'loss': float}]] (only the keys all of them have)  (B: not built yet)
# (B: a dict with another plain type, like {'loss': 0.6} here, a dict[str, float], makes it plain)

In [ ]:
# 3. A write inside, to a dict the checker can't pin down
some = [{'loss': 0.4, 'tag': 'a'}, {'loss': 0.5, 'tag': 'b'}]
for i in range(len(some)):
    some[i]['loss'] = 'n/a'
some  # A: list[dict[{'loss': float, 'tag': str}]] (stale: 'loss' is 'n/a' now)
#       B: list[dict[str, Unknown]] (plain, like scores[subject] = 100 on a plain dict)
some[0]['loss']  # A: float (wrong)   B: Unknown

In [ ]:
# 4. A write inside, through literal keys: the checker knows which dict
by_run = {name: {'best': 0.4, 'epoch': 3} for name in ['a', 'b']}
by_run['a']['best'] = 'n/a'
by_run  # A: 'a' still dict[{'best': float, 'epoch': int}] (stale)
#         B: 'a' is dict[{'best': str, 'epoch': int}], 'b' unchanged (like user['name'] = 2)
by_run['b']['best']  # A and B: float

In [ ]:
# 5. Writes to the dict's own name: the same in both
own = {name: {'best': 0.4} for name in ['a', 'b']}
own['c'] = {'best': 0.1}
own  # A and B: 'c' added
del own['a']
own  # A and B: 'a' removed

In [ ]:
# 6. Writes the checker can't see: stale in both (the known hole, as rename(user) for a plain dict)
seen = [{'loss': 0.4, 'tag': 'a'}]
for r in seen:
    r['loss'] = 'n/a'
seen[0]['loss']  # A and B: float, but it's 'n/a'


def reset(d: dict[str, float | str]) -> None:
    d['loss'] = 'n/a'


called = [{'loss': 0.4, 'tag': 'a'}]
reset(called[0])
called[0]['loss']  # A and B: float, but it's 'n/a'

## Summary

| | A. As built | B. Tracked where possible |
|---|---|---|
| Rule | dicts inside keep the keys they were built with | the plain-dict rules, applied inside containers |
| `rows.append(same keys)` | unchanged | unchanged |
| `rows.append(other keys)` | stale | the keys all of them have |
| `rows[i]['k'] = v` | stale (wrong type) | plain (`Unknown`) |
| `by['a']['k'] = v` | stale | `'a'` updated |
| `for r in rows: r['k'] = v`, `f(rows[0])` | stale | stale |
| Wrong types shown | after any write inside | only after writes it can't see |
| Code in the fork | less (write tracking removed) | today's + the append join |
